# Import libraries

In [1]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from pandas_gbq import to_gbq
import os

print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Set the environment variable for Google Cloud credentials
# Place the path in which the .json file is located.

# Example (if .json is located in the same directory with the notebook)
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "at-arch-416714-6f9900ec7.json"

# -- YOUR CODE GOES BELOW THIS LINE
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\roxan\AppData\Roaming\gcloud\application_default_credentials.json"
# -- YOUR CODE GOES ABOVE THIS LINE

In [3]:
# Set your Google Cloud project ID and BigQuery dataset details

project_id = 'project-401f4646-3663-4125-aaa' # Edit with your project id
dataset_id = 'staging_db' # Modify the necessary schema name: staging_db, reporting_db etc.
table_id = 'stg_address' # Modify the necessary table name: stg_customer, stg_city etc.

# SQL Query

In [4]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Define your SQL query here
query = """
with base as (
  select *
  from `project-401f4646-3663-4125-aaa.pagila_productionpublic.address` 
  )

  , final as (
    select
          address_id
        , address  as address_address
        , address2 as address_address2
        , district as address_district
        , city_id as address_city_id
        , postal_code as address_postal_code
        , phone as address_phone
        , last_update as address_last_update
   FROM base
  )

  select * from final
"""

# Execute the query and store the result in a dataframe
df = client.query(query).to_dataframe()

# Explore some records
df.head()

C:\Users\roxan\PycharmProjects\Pagila Analytics Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,address_id,address_address,address_address2,address_district,address_city_id,address_postal_code,address_phone,address_last_update
0,3,23 Workhaven Lane,NaN,Alberta,300,,14033335568,2022-02-15 09:45:30+00:00
1,4,1411 Lillydale Drive,NaN,QLD,576,,6172235589,2022-02-15 09:45:30+00:00
2,2,28 MySQL Boulevard,NaN,QLD,576,,,2022-02-15 09:45:30+00:00
3,1,47 MySakila Drive,NaN,Alberta,300,,,2022-02-15 09:45:30+00:00
4,11,900 Santiago de Compostela Parkway,,Central Serbia,280,93896,716571220373,2022-02-15 09:45:30+00:00


# Write to BigQuery

In [5]:
# Define the full table ID
full_table_id = f"{project_id}.{dataset_id}.{table_id}"

# Define table schema based on the project description

schema = [
    bigquery.SchemaField('address_id', 'INTEGER'),
    bigquery.SchemaField('address_address', 'STRING'),
    bigquery.SchemaField('address_address2', 'STRING'),
    bigquery.SchemaField('address_district', 'STRING'),
    bigquery.SchemaField('address_city_id', 'INTEGER'),
    bigquery.SchemaField('address_postal_code', 'STRING'),
    bigquery.SchemaField('address_phone', 'STRING'),
    bigquery.SchemaField('address_last_update', 'DATETIME')
    ]


In [6]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Check if the table exists
def table_exists(client, full_table_id):
    try:
        client.get_table(full_table_id)
        return True
    except Exception:
        return False

# Write the dataframe to the table (overwrite if it exists, create if it doesn't)
if table_exists(client, full_table_id):
    # If the table exists, overwrite it
    destination_table = f"{dataset_id}.{table_id}"
    # Write the dataframe to the table (overwrite if it exists)
    to_gbq(df, destination_table, project_id=project_id, if_exists='replace')
    print(f"Table {full_table_id} exists. Overwritten.")
else:
    # If the table does not exist, create it
    job_config = bigquery.LoadJobConfig(schema=schema)
    job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
    job.result()  # Wait for the job to complete
    print(f"Table {full_table_id} did not exist. Created and data loaded.")

Table project-401f4646-3663-4125-aaa.staging_db.stg_address did not exist. Created and data loaded.


In [7]:
# Converting  i.pynb file to .py python executable file. 

!python -m jupyter nbconvert stg_address.ipynb --to python

[NbConvertApp] Converting notebook stg_address.ipynb to python
[NbConvertApp] Writing 4170 bytes to stg_address.py


In [8]:
!python -m pip install nbconvert


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!python -m pip install nbconvert -U


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
